In [1]:
import h5py
import pandas as pd
import numpy as np
import yaml
import os
from tkinter import Tk, filedialog

def process_hdf5_files(file_list, output_path):
    all_dfs = []

    for i, filename in enumerate(file_list):
        print(f"\nProcessing: {filename}")
        print(f"   y-shift applied: {i * 10}")

        # Load full dataset from Picasso file
        with h5py.File(filename, "r") as f:
            locs = f["locs"][:]          # structured numpy array
            df = pd.DataFrame(locs)      # convert to DataFrame

        # Apply y-shift for this file
        df["y"] = df["y"] + i * 12

        # Filter probability > 0.95 if exists
        if "prob" in df.columns:
            before = len(df)
            df = df[df["prob"] > 0.95]
            after = len(df)
            print(f"   Filtered: {after}/{before} kept ({after/before*100:.2f}%)")
        else:
            print("   Warning: 'prob' column not found -> skipping filtering")

        df.reset_index(drop=True, inplace=True)
        all_dfs.append(df)

    # Combine all dataframes
    combined_df = pd.concat(all_dfs, ignore_index=True)
    print(f"\nTotal localizations after combining: {len(combined_df)}")

    # Convert back to structured array for Picasso
    locs_out = pd.DataFrame.to_records(combined_df)

    # Determine filenames
    hdf5_out = output_path
    yaml_out = os.path.splitext(output_path)[0] + ".yaml"  # same name as HDF5

    # Write combined HDF5
    with h5py.File(hdf5_out, "w") as f:
        f.create_dataset("locs", data=locs_out)
    print(f"\nSaved combined HDF5 to: {hdf5_out}")

    # Create YAML metadata
    frames = int(combined_df["frame"].max() + 1)
    yaml_dict = {
        "Frames": frames,
        "Height": 512,
        "Width": 512,
        "Pixelsize": 97,
    }

    # Write YAML metadata
    with open(yaml_out, "w") as f:
        yaml.dump(yaml_dict, f)
    print(f"Saved YAML metadata to: {yaml_out}")

    return combined_df


# --- MAIN EXECUTION ---

print("Select the HDF5 files you want to combine")

Tk().withdraw()  # Hide Tkinter window

# Select input files
file_paths = filedialog.askopenfilenames(
    title="Select HDF5 files",
    filetypes=[("HDF5 files", "*.hdf5"), ("All files", "*.*")],
)

file_paths = list(file_paths)
print(f"\nNumber of files selected: {len(file_paths)}")
if len(file_paths) == 0:
    print("No files selected. Exiting.")
else:
    print("\nFiles selected:")
    for f in file_paths:
        print("  -", f)

    # Ask where to save the merged HDF5 file
    output_path = filedialog.asksaveasfilename(
        title="Save merged HDF5 file as",
        defaultextension=".hdf5",
        filetypes=[("HDF5 files", "*.hdf5")],
    )

    if not output_path:
        print("No output file selected. Exiting.")
    else:
        process_hdf5_files(file_paths, output_path)


Select the HDF5 files you want to combine

Number of files selected: 0
No files selected. Exiting.


In [8]:
import h5py
from tkinter import Tk, filedialog

# Hide the empty Tkinter window
Tk().withdraw()

# Ask the user to select ONE HDF5 file
file_path = filedialog.askopenfilename(
    title="Select an HDF5 file",
    filetypes=[("HDF5 files", "*.hdf5"), ("All files", "*.*")]
)

if not file_path:
    print("No file selected. Exiting.")
else:
    print("Selected file:", file_path)

    # Open the HDF5 file
    with h5py.File(file_path, "r") as f:
        print("Top-level groups/datasets:", list(f.keys()))


Selected file: //serverme41/DataSchnorrer2/EqpSchnorrer/Aditya/Experiments/Zasp and actinin/ZaspActinin/20230726_sample1_1_150Actinin_P1_A647_2nM_XmW_locs_filter_render_picked_DBSCAN_2_#1.hdf5
Top-level groups/datasets: ['locs']


In [5]:
import h5py
import pandas as pd

print("\n=== Column inspection for each loaded file ===")

file_columns = []

# 1) Read columns from each file
for idx, filename in enumerate(file_paths):
    with h5py.File(filename, "r") as f:
        locs = f["locs"][:]
        df_temp = pd.DataFrame(locs)

    print(f"\nFile {idx+1}: {filename}")
    print("Columns:", list(df_temp.columns))
    file_columns.append((filename, list(df_temp.columns)))


print("\n=== Checking for duplicates during merging ===")

merged_cols = pd.Index([])

# 2) Detect duplicates incrementally
for idx, (filename, cols) in enumerate(file_columns):
    new_cols = merged_cols.append(pd.Index(cols))

    dupes = new_cols[new_cols.duplicated()]
    if len(dupes) > 0:
        print(f"\n⚠️ DUPLICATE columns introduced when adding file {idx+1}:")
        print(f"   {filename}")
        for d in dupes:
            print("   -", d)
    else:
        print(f"File {idx+1} added OK (no duplicates): {filename}")

    merged_cols = new_cols



=== Column inspection for each loaded file ===

File 1: Y:/EqpSchnorrer/Aditya/Experiments/Proj RNAi/Control/20250810_sample1_1_ProjRNAi_Ctrl_Strn_P3_Cy3B_Phall488_XmW_picasso_20230418_3_filter_render_picked_DBSCAN_#1.hdf5
Columns: ['index', 'frame', 'x', 'y', 'z', 'photons', 'sx', 'sy', 'sz', 'bg', 'lpx', 'lpy', 'lpz', 'ellipticity', 'net_gradient', 'prob']

File 2: Y:/EqpSchnorrer/Aditya/Experiments/Proj RNAi/Control/20250810_sample1_1_ProjRNAi_Ctrl_Strn_P3_Cy3B_Phall488_XmW_picasso_20230418_3_filter_render_picked_DBSCAN_#2.hdf5
Columns: ['index', 'frame', 'x', 'y', 'z', 'photons', 'sx', 'sy', 'sz', 'bg', 'lpx', 'lpy', 'lpz', 'ellipticity', 'net_gradient', 'prob']

File 3: Y:/EqpSchnorrer/Aditya/Experiments/Proj RNAi/Control/20250810_sample1_1_ProjRNAi_Ctrl_Strn_P3_Cy3B_Phall488_XmW_picasso_20230418_3_filter_render_picked_DBSCAN_#3.hdf5
Columns: ['index', 'frame', 'x', 'y', 'z', 'photons', 'sx', 'sy', 'sz', 'bg', 'lpx', 'lpy', 'lpz', 'ellipticity', 'net_gradient', 'prob']

File 4: Y

In [5]:
import h5py
import pandas as pd
import numpy as np
import yaml
import os
from tkinter import Tk, filedialog

def process_hdf5_files(file_list, output_path):
    all_dfs = []

    for i, filename in enumerate(file_list):
        print(f"\nProcessing: {filename}")
        print(f"   y-shift applied: {i * 12}")

        # Load Picasso file
        with h5py.File(filename, "r") as f:
            locs = f["locs"][:]                  # structured numpy array
            df = pd.DataFrame(locs)              # convert to DataFrame

        # Remove unnecessary index column (prevents dtype collision)
        if "index" in df.columns:
            df.drop(columns=["index"], inplace=True)

        # Apply y-shift
        df["y"] = df["y"] + i * 12

        # Filter probability
        if "prob" in df.columns:
            before = len(df)
            df = df[df["prob"] > 0.95]
            after = len(df)
            print(f"   Filtered: {after}/{before} kept ({after/before*100:.2f}%)")
        else:
            print("   Warning: 'prob' column not found → skipping filtering")

        df.reset_index(drop=True, inplace=True)
        all_dfs.append(df)

    # Combine all datasets
    combined_df = pd.concat(all_dfs, ignore_index=True)
    print(f"\nTotal localizations after combining: {len(combined_df)}")

    # Sort columns into a clean Picasso-friendly order
    order = [
        "frame", "x", "y", "z",
        "photons", "sx", "sy", "sz",
        "bg", "lpx", "lpy", "lpz",
        "ellipticity", "net_gradient", "prob"
    ]
    available = [c for c in order if c in combined_df.columns]
    combined_df = combined_df[available]

    # Convert back to Picasso locs format
    locs_out = combined_df.to_records(index=False)   # ← FIXED

    # Determine filenames
    hdf5_out = output_path
    yaml_out = os.path.splitext(output_path)[0] + ".yaml"

    # Write HDF5
    with h5py.File(hdf5_out, "w") as f:
        f.create_dataset("locs", data=locs_out)
    print(f"\nSaved combined HDF5 to: {hdf5_out}")

    # YAML metadata
    frames = int(combined_df["frame"].max() + 1) if "frame" in combined_df else 1
    yaml_dict = {
        "Frames": frames,
        "Height": 512,
        "Width": 512,
        "Pixelsize": 97,
    }

    with open(yaml_out, "w") as f:
        yaml.dump(yaml_dict, f)
    print(f"Saved YAML metadata to: {yaml_out}")

    return combined_df


# --- MAIN EXECUTION ---

print("Select the HDF5 files you want to combine")

Tk().withdraw()

# Select input files
file_paths = filedialog.askopenfilenames(
    title="Select HDF5 files",
    filetypes=[("HDF5 files", "*.hdf5"), ("All files", "*.*")],
)

file_paths = list(file_paths)
print(f"\nNumber of files selected: {len(file_paths)}")

if len(file_paths) == 0:
    print("No files selected. Exiting.")
else:
    print("\nFiles selected:")
    for f in file_paths:
        print("  -", f)

    # Ask where to save the merged HDF5 file
    output_path = filedialog.asksaveasfilename(
        title="Save merged HDF5 file as",
        defaultextension=".hdf5",
        filetypes=[("HDF5 files", "*.hdf5")],
    )

    if not output_path:
        print("No output file selected. Exiting.")
    else:
        process_hdf5_files(file_paths, output_path)


Select the HDF5 files you want to combine

Number of files selected: 3

Files selected:
  - E:/2026_02_18_keep/Analysis/20260218_sample3_1_Nb37_P3_Cy3B_2nM_Phall488_XmW_picasso_20230418_3_filter_render_picked_DBSCAN_2_#3.hdf5
  - E:/2026_02_18_keep/Analysis/20260218_sample3_1_Nb37_P3_Cy3B_2nM_Phall488_XmW_picasso_20230418_3_filter_render_picked_DBSCAN_2_#2.hdf5
  - E:/2026_02_18_keep/Analysis/20260218_sample3_1_Nb37_P3_Cy3B_2nM_Phall488_XmW_picasso_20230418_3_filter_render_picked_DBSCAN_2_#1.hdf5

Processing: E:/2026_02_18_keep/Analysis/20260218_sample3_1_Nb37_P3_Cy3B_2nM_Phall488_XmW_picasso_20230418_3_filter_render_picked_DBSCAN_2_#3.hdf5
   y-shift applied: 0
   Filtered: 10010/10010 kept (100.00%)

Processing: E:/2026_02_18_keep/Analysis/20260218_sample3_1_Nb37_P3_Cy3B_2nM_Phall488_XmW_picasso_20230418_3_filter_render_picked_DBSCAN_2_#2.hdf5
   y-shift applied: 12
   Filtered: 7585/7585 kept (100.00%)

Processing: E:/2026_02_18_keep/Analysis/20260218_sample3_1_Nb37_P3_Cy3B_2nM_Phal